In [87]:
import pandas as pd

In [88]:
df = pd.read_csv('Airplanes_modified.csv')

In [89]:
us_regions = {
    'South': [
        'TX', 'VA', 'MS', 'FL', 'AR', 'LA', 'OK', 'NC', 
        'KY', 'AL', 'TN', 'MD', 'SC', 'WV', 'GA'
    ],
    'West': [
        'NV', 'CA', 'OR', 'AZ', 'WA', 'UT', 'NM', 'ID', 
        'CO', 'WY', 'HI', 'MT', 'AK'
    ],
    'Midwest': [
        'IN', 'MO', 'IL', 'NE', 'OH', 'MI', 'WI', 'KS', 
        'IA', 'MN', 'SD', 'ND'
    ],
    'Northeast': [
        'NY', 'NH', 'PA', 'RI', 'CT', 'NJ', 'VT', 'ME', 'MA'
    ],
    'Other': [
        'PR', 'VI' 
    ]
}

In [90]:
print(df.head())

   Month  DayofMonth  DayOfWeek  DepTime  CRSDepTime  CRSArrTime  \
0      1           1          2   2400.0          10         737   
1      1           1          2      5.0          15         823   
2      1           1          2     19.0          25         709   
3      1           1          2     45.0          25         535   
4      1           1          2     26.0          30         444   

   CRSElapsedTime  ArrDelay  Distance  previous_dep_delay  \
0           267.0      -3.0      1979                82.0   
1           308.0     -19.0      2521                 7.0   
2           284.0     -24.0      2153                 0.0   
3           190.0      12.0      1431                 0.0   
4           194.0      24.0      1449                 0.0   

   scheduled_turnaround origin_state  origin_lat  origin_long dest_state  \
0                 248.0           CA   33.942536  -118.408074         MI   
1                 980.0           CA   38.695422  -121.590767         NY

## move flights within region to one airport on a specific day

In [91]:
# region_states = {state: region for region, states in us_regions.items() for state in states}
# df['region'] = df['dest_state'].map(region_states)

In [92]:
region_states = {state: region for region, states in us_regions.items() for state in states}
origin_region = df['origin_state'].map(region_states)
dest_region   = df['dest_state'].map(region_states)
df['region'] = df.apply(
    lambda r: dest_region[r.name] if origin_region[r.name] == dest_region[r.name] else 'NA',
    axis=1
)

In [93]:
df['region'].value_counts()

region
NA           1098022
South         512119
West          511752
Midwest       150917
Northeast      45926
Other            385
Name: count, dtype: int64

In [94]:
df.Month.unique()

array([1, 2, 3, 4])

In [95]:
mask = ((df.Month==2) & (df.DayofMonth.between(15,20)) & (df.region=='Northeast'))

In [96]:
df[mask]['dest_state'].value_counts()

dest_state
NY    806
MA    481
PA    405
NJ    267
RI     75
ME     66
VT     63
NH     59
CT     45
Name: count, dtype: int64

In [97]:
df.loc[mask, 'dest_state'] = 'MA'

In [98]:
df.loc[mask, 'previous_dep_delay'] += 30

In [99]:
df.loc[mask, 'ArrDelay'] += 30

In [100]:
df[mask].head()

,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,CRSArrTime,CRSElapsedTime,ArrDelay,Distance,previous_dep_delay,scheduled_turnaround,origin_state,origin_lat,origin_long,dest_state,dest_lat,dest_long,region
844547,2,15,5,524.0,530,655,85.0,28.0,280,30.0,1440.0,MA,42.364348,-71.005179,MA,39.871953,-75.241141,Northeast
844598,2,15,5,549.0,550,701,71.0,20.0,187,30.0,1440.0,MA,42.364348,-71.005179,MA,40.639751,-73.778926,Northeast
844599,2,15,5,548.0,550,717,87.0,30.0,301,42.0,1305.0,NY,42.940525,-78.732167,MA,40.639751,-73.778926,Northeast
844662,2,15,5,557.0,600,724,84.0,33.0,284,30.0,1440.0,ME,43.646167,-70.308750,MA,40.692497,-74.168661,Northeast
844677,2,15,5,557.0,600,718,78.0,6.0,209,30.0,1440.0,NY,43.111187,-76.106311,MA,40.639751,-73.778926,Northeast


In [101]:
mask2 = ((df.Month==2) & (df.DayofMonth.between(8,10)) & (df.region=='South'))

In [102]:
df[mask2]['dest_state'].value_counts()

dest_state
TX    3554
FL    1917
GA    1887
NC     843
VA     732
TN     702
LA     443
KY     439
MD     351
AL     285
OK     223
SC     207
AR     178
MS     159
WV      18
Name: count, dtype: int64

In [103]:
df.loc[mask2, 'dest_state'] = 'GA'

In [104]:
df.loc[mask2, 'ArrDelay'] += 30

In [105]:
df.loc[mask2, 'previous_dep_delay'] += 30

In [106]:
df.to_csv('Airplanes_synthetic.csv', index=False)

In [107]:
df.head(10)

,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,CRSArrTime,CRSElapsedTime,ArrDelay,Distance,previous_dep_delay,scheduled_turnaround,origin_state,origin_lat,origin_long,dest_state,dest_lat,dest_long,region
0,1,1,2,2400.0,10,737,267.0,-3.0,1979,82.0,248.0,CA,33.942536,-118.408074,MI,42.212059,-83.348836,NA
1,1,1,2,5.0,15,823,308.0,-19.0,2521,7.0,980.0,CA,38.695422,-121.590767,NY,40.639751,-73.778926,NA
2,1,1,2,19.0,25,709,284.0,-24.0,2153,0.0,1440.0,AZ,33.434167,-112.008056,NY,40.639751,-73.778926,NA
3,1,1,2,45.0,25,535,190.0,12.0,1431,0.0,1440.0,CA,38.695422,-121.590767,TX,32.895951,-97.037200,NA
4,1,1,2,26.0,30,444,194.0,24.0,1449,0.0,1440.0,AK,61.174320,-149.996186,WA,47.448982,-122.309313,West
5,1,1,2,29.0,30,825,295.0,-27.0,2248,0.0,1440.0,NV,36.080361,-115.152333,NY,40.639751,-73.778926,NA
6,1,1,2,33.0,30,605,215.0,11.0,1536,0.0,1440.0,CA,33.942536,-118.408074,MN,44.880547,-93.216922,NA
7,1,1,2,21.0,30,831,301.0,-23.0,2430,0.0,1440.0,CA,34.056000,-117.601194,NY,40.639751,-73.778926,NA
8,1,1,2,33.0,35,603,208.0,-12.0,1635,0.0,1440.0,CA,37.619002,-122.374843,TX,29.980472,-95.339722,NA
9,1,1,2,30.0,35,420,165.0,-15.0,1189,0.0,1440.0,FL,28.428889,-81.316028,PR,18.439417,-66.001833,NA
